In [1]:
!wget https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip -O data_original.zip
!unzip data_original.zip

--2023-11-19 10:34:13--  https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/irsafilo/KION_DATASET/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip [following]
--2023-11-19 10:34:14--  https://raw.githubusercontent.com/irsafilo/KION_DATASET/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 78795295 (75M) [application/zip]
Saving to: ‘data_original.zip’

data_original.zip   100%[===================>]  75.14M   242MB/s    in 0.3s    

In [2]:
!pip install rectools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.0/99.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.3 MB/s eta 0:00:00


In [64]:
import time

import typing
from collections.abc import KeysView
from copy import deepcopy
import numpy as np
import pandas as pd
from tqdm import tqdm
from pprint import pprint
import matplotlib.pyplot as plt
import seaborn as sns

from implicit.nearest_neighbours import TFIDFRecommender

from rectools import Columns

from rectools.dataset import Interactions, Dataset
from rectools.metrics import Precision, Recall, MeanInvUserFreq, Serendipity, calc_metrics
from rectools.metrics.ranking import MAP, MRR, NDCG
from rectools.models import RandomModel, PopularModel
from rectools.model_selection import TimeRangeSplitter

### Расчет метрик

In [65]:

def calculate_metrics(models, metrics, splitter, K, data):
    """
    Функция для расчёта метрик с использованием кросс-валидации.
    Аргументы:
    - models (dict): Словарь с инициализированными моделями.
    - metrics (dict): Словарь с инициализированными метриками.
    - splitter (Splitter object): Объект Splitter для кросс-валидации.
    - K (int): Количество рекомендаций для генерации.
    - data (DataFrame): Исходный датасет.

    Возвращает:
    - DataFrame: Усреднённые результаты метрик по моделям и метрикам.
    """
    interactions = Interactions(data)
    splitter.get_test_fold_borders(interactions)
    dataset = Dataset.construct(data)
    results = []
    fold_iterator = splitter.split(interactions, collect_fold_stats=True)

    for train_ids, test_ids, fold_info in tqdm((fold_iterator), total=splitter.n_splits):
        print(f"\n==================== Fold {fold_info['i_split']}")
        pprint(fold_info)

        df_train = interactions.df.iloc[train_ids]
        dataset = Dataset.construct(df_train)

        df_test = interactions.df.iloc[test_ids][Columns.UserItem]
        test_users = np.unique(df_test[Columns.User])

        # Catalog is set of items that we recommend.
        # Sometimes we recommend not all items from train.
        catalog = df_train[Columns.Item].unique()

        for model_name, model in models.items():
            model.fit(dataset)
            recos = model.recommend(
                users=test_users,
                dataset=dataset,
                k=K,
                filter_viewed=True,
            )
            metric_values = calc_metrics(
                metrics,
                reco=recos,
                interactions=df_test,
                prev_interactions=df_train,
                catalog=catalog,
            )
            res = {"fold": fold_info["i_split"], "model": model_name}
            res.update(metric_values)
            results.append(res)

    pivot_results = pd.DataFrame(results).drop(columns="fold").groupby(["model"], sort=False).agg(["mean", "std"])
    mean_metric_subset = [(metric, agg) for metric, agg in pivot_results.columns if agg == 'mean']
    pivot_results = (
        pivot_results.style
        .highlight_min(subset=mean_metric_subset, color='lightcoral', axis=0)
        .highlight_max(subset=mean_metric_subset, color='lightgreen', axis=0)
    )
    return pivot_results

In [98]:
def visualize_recommendations(model, data, user_ids, item_data, K):
    reco = model.recommend(
        users=user_ids,
        dataset=Dataset.construct(data),
        k=K,
        filter_viewed=True,
    )

    for user in user_ids:
        print(f"Visualization for User ID: {user}")

        # История просмотров пользователя
        user_history = (
            data[data.user_id == user]
            .merge(item_data, on='item_id')
            .sort_values(by='datetime', ascending=False)
        )

        # Рекомендации для пользователя
        user_recommendations = (
            reco[reco.user_id == user]
            .merge(item_data, on='item_id')
        )

        # Вывод истории просмотров
        print("\nUser History:")
        display(user_history)

        # Вывод рекомендаций
        print("\nUser Recommendations:")
        display(user_recommendations)

### Тестирование

In [84]:
models = {
    'RandomModel': RandomModel(random_state=32),
    'PopularModel': PopularModel()
}

In [85]:
metrics = {
    "Precision@1": Precision(k=1),
    "Precision@5": Precision(k=5),
    "Precision@10": Precision(k=10),

    "Recall@1": Recall(k=1),
    "Recall@5": Recall(k=5),
    "Recall@10": Recall(k=10),

    "MAP@1": MAP(k=1),
    "MAP@5": MAP(k=5),
    "MAP@10": MAP(k=10),

    "MRR@1": MRR(k=1),
    "MRR@5": MRR(k=5),
    "MRR@10": MRR(k=10),

    "NDCG@1": NDCG(k=1),
    "NDCG@5": NDCG(k=5),
    "NDCG@10": NDCG(k=10),

    "Novelty@1": MeanInvUserFreq(k=1),
    "Novelty@5": MeanInvUserFreq(k=5),
    "Novelty@10": MeanInvUserFreq(k=10),

    "Serendipity@1": Serendipity(k=1),
    "Serendipity@5": Serendipity(k=5),
    "Serendipity@10": Serendipity(k=10),
}

In [86]:
splitter = TimeRangeSplitter(
    test_size="7D",
    n_splits=3,
    filter_already_seen=True,
    filter_cold_items=True,
    filter_cold_users=True,
)

In [87]:
path_data = "data_original"

interactions_df = pd.read_csv(
    path_data + "/" + "interactions.csv",
    sep=",",
    nrows=100_000,
    names=[Columns.User, Columns.Item, Columns.Datetime, Columns.Weight, Columns.Score],
)

interactions_df = interactions_df.iloc[1:]
interactions_df = interactions_df[[Columns.User, Columns.Item, Columns.Datetime, Columns.Score]]
interactions_df[Columns.Weight] = 1.0

interactions_df["item_id"] = interactions_df["item_id"].astype("int")
interactions_df[Columns.User] = interactions_df[Columns.User].astype("int")

print(interactions_df.shape)
interactions_df.head()

(99999, 5)


,user_id,item_id,datetime,score,weight
1,176549,9506,2021-05-11,72.0,1.0
2,699317,1659,2021-05-29,100.0,1.0
3,656683,7107,2021-05-09,0.0,1.0
4,864613,7638,2021-07-05,100.0,1.0
5,964868,9506,2021-04-30,100.0,1.0


In [88]:
calculate_metrics(models, metrics, splitter, K=10, data=interactions_df)

  0%|          | 0/3 [00:00<?, ?it/s]


==================== Fold 0
{'end': Timestamp('2021-08-09 00:00:00', freq='7D'),
 'i_split': 0,
 'start': Timestamp('2021-08-02 00:00:00', freq='7D'),
 'test': 1015,
 'test_items': 633,
 'test_users': 962,
 'train': 77971,
 'train_items': 5784,
 'train_users': 66340}


 33%|███▎      | 1/3 [00:01<00:03,  1.57s/it]


==================== Fold 1
{'end': Timestamp('2021-08-16 00:00:00', freq='7D'),
 'i_split': 1,
 'start': Timestamp('2021-08-09 00:00:00', freq='7D'),
 'test': 1122,
 'test_items': 703,
 'test_users': 1045,
 'train': 84922,
 'train_items': 5986,
 'train_users': 71962}


 67%|██████▋   | 2/3 [00:02<00:01,  1.34s/it]


==================== Fold 2
{'end': Timestamp('2021-08-23 00:00:00', freq='7D'),
 'i_split': 2,
 'start': Timestamp('2021-08-16 00:00:00', freq='7D'),
 'test': 1259,
 'test_items': 779,
 'test_users': 1169,
 'train': 92262,
 'train_items': 6146,
 'train_users': 77826}


100%|██████████| 3/3 [00:03<00:00,  1.31s/it]


In [89]:
%%time
items_df = pd.read_csv(
    path_data + "/" + "items.csv",
    sep=",",
)
items_df.head()

CPU times: user 1.01 s, sys: 93.2 ms, total: 1.1 s
Wall time: 2.29 s


,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [99]:
user_ids = [176549, 699317, 656683]

# Проведение визуального анализа
for model_name, model in models.items():
    print(f"Visualization for {model_name}")
    res =  visualize_recommendations(model, interactions_df, user_ids, items_df, K=10)
    print(res)

Visualization for RandomModel
Visualization for User ID: 176549

User History:


,user_id,item_id,datetime,score,weight,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
1,176549,15469,2021-05-25,100.0,1.0,film,Безбашенная пуля,Hollow Point,2019.0,"боевики, триллеры",США,NaN,16.0,NaN,Дэниэл Дзирилли,"Люк Госс, Дилан Джей, Жужу Чан, Джей Мор, Билл...","Хэнк Кормак — адвокат и мститель, который возг...","месть, дружинники, Дочь, 2019, соединенные шта..."
2,176549,9164,2021-05-20,100.0,1.0,film,ВАЛЛ-И,WALL·E,2008.0,"фантастика, мультфильм, приключения",США,NaN,0.0,NaN,Эндрю Стэнтон,"Бен Бертт, Элисса Найт, Джефф Гарлин, Фред Уил...","В далёком будущем, когда Земля стала необитаем...","мусор, космические путешествия, антиутопия, од..."
3,176549,12250,2021-05-18,58.0,1.0,film,Джон Уик 2,John Wick: Chapter 2,2017.0,"боевики, триллеры, криминал","США, Гонконг",NaN,18.0,NaN,Чад Стахелски,"Иэн МакШейн, Лоренс Фишбёрн, Дэвид Патрик Келл...",Когда бывший коллега Джона решает взять под св...,"Италия, пистолет, крыша, вечеринка, продолжени..."
0,176549,9506,2021-05-11,72.0,1.0,film,Холодное сердце,Frozen,2013.0,"фэнтези, мультфильм, музыкальные",США,NaN,0.0,NaN,"Крис Бак, Дженнифер Ли","Кристен Белл, Идина Мензел, Джонатан Грофф, Дж...","Когда сбывается древнее предсказание, и короле...","королева, мюзикл, принцесса, предательство, сн..."



User Recommendations:


,user_id,item_id,score,rank,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,176549,11312,10,1,film,Сумасшедшая любовь,Words on Bathroom Walls,2020.0,мелодрамы,США,NaN,12.0,NaN,Тор Фройденталь,"Чарли Пламмер, Молли Паркер, Энди Гарсиа, Уолт...",До последнего года учебы в школе Адаму удаётся...,"шизофрения, кулинария, повествование, старшая ..."
1,176549,6214,9,2,film,Дикие предки,Early Man,2018.0,"мультфильм, приключения, спорт, фэнтези, комедии","Великобритания, Франция",NaN,6.0,NaN,Ник Парк,"Тимоти Сполл, Эдди Редмэйн, Том Хиддлстон, Мэй...",Каменный век и даже более поздние цивилизации ...,"каменный век, добыча полезных ископаемых, спор..."
2,176549,2220,8,3,film,С днём смерти,Death of Me,2020.0,"ужасы, детективы","США, Таиланд",NaN,18.0,NaN,Даррен Линн Боусман,"Мэгги Кью, Люк Хемсворт, Александра Эссоу, Кэт...",Супруги-американцы Кристин и Нейл проводят отп...,"самоубийство, фотограф, паспорт, галлюцинация,..."
3,176549,4008,7,4,film,Запретная любовь,The Edge of Love,2008.0,"драмы, биография, военные, мелодрамы",Великобритания,NaN,16.0,NaN,Джон Мэйбери,"Кира Найтли, Сиенна Миллер, Киллиан Мёрфи, Мэт...","Сложные межличностные отношения четырех людей,...","биография, поэт, 2008, соединенное королевство..."
4,176549,5105,6,5,film,Жесткие правила,Lust For Young Busts,2011.0,для взрослых,"Нидерланды, Великобритания",NaN,21.0,NaN,"Денис Марти, Газман, Странж Лове","Блэк Анджелика, Алина Хенесси, Мэнди Ди, Саша ...",Героини нового фильма режиссера Газзмана порад...,"Жесткие правила, Lust For Young Busts, Блэк Ан..."
5,176549,4729,5,6,film,Прорвёмся,NaN,2018.0,драмы,Россия,0.0,18.0,NaN,Илья Фарфель,"Назар Сафонов, Валерия Дмитриева, Илья Коробко...",Брат с сестрой мечтают поскорей вырваться из с...,"Прорвёмся, 2018, Россия"
6,176549,9142,4,7,film,Меланхолия,Melancholia,2011.0,"драмы, фантастика, триллеры","Дания, Швеция, Франция, Германия",NaN,16.0,NaN,Ларс фон Триер,"Кирстен Данст, Шарлотта Рэмплинг, Стеллан Скар...","События фильма разворачиваются в дни, которые ...","самоубийство, депрессия, нигилизм, организатор..."
7,176549,12096,3,8,series,Апостол,Apostol,2008.0,"боевики, историческое, военные",Россия,NaN,16.0,NaN,"Юрий Мороз, Николай Лебедев, Геннадий Сидоров","Евгений Миронов, Николай Фоменко, Дарья Мороз,...",В начале войны немцы забрасывают в СССР своего...,NaN
8,176549,13144,2,9,film,Рыцари Ньюгейта,Knights of Newgate,2021.0,"зарубежные, фэнтези, приключения",Великобритания,NaN,16.0,NaN,NaN,"Ви Вимолмал, Лара Доре, Роберт Реина",Действие фэнтезийной картины разворачивается в...,"Рыцари, Ньюгейта, 2021, Великобритания, интриг..."
9,176549,15815,1,10,film,Садко,Sadko,2017.0,"мультфильм, приключения, комедии",Россия,NaN,6.0,NaN,"М. Волков, В. Мухаметзянов","Тимур Гарипов, Любовь Аксенова, Екатерина Варн...","Хоть беги, хоть не беги, а от любви нигде не с...","2017, россия, садко"


Visualization for User ID: 699317

User History:


...

Visualization for User ID: 656683

User History:


,user_id,item_id,datetime,score,weight,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,656683,7107,2021-05-09,0.0,1.0,series,Девятаев,V2. Escape from Hell,2021.0,"драмы, военные, приключения",Россия,NaN,12.0,NaN,Тимур Бекмамбетов,"Павел Прилучный, Павел Чинарёв, Тимофей Трибун...",Военно-исторический блокбастер от режиссёров Т...,NaN



User Recommendations:


,user_id,item_id,score,rank,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,656683,4398,3563.0,1,series,Муж на час,NaN,2014.0,"мелодрамы, комедии",Украина,NaN,12.0,NaN,Анатолий Матешко,"Ярослав Бойко, Мария Куликова, Юрий Горбунов, ...","Главный герой работал в НИИ, конструировал нов...",NaN
1,656683,11754,3266.0,2,film,Kingsman: Секретная служба,Kingsman: The Secret Service,2015.0,"боевики, криминал, приключения, комедии","Великобритания, США",NaN,18.0,NaN,Мэттью Вон,"Тэрон Эджертон, Колин Фёрт, Сэмюэл Л. Джексон,...","Эггси — молодой парень, который прошел службу ...","шпион, великобритания, секретная организация, ..."
2,656683,3813,2198.0,3,film,Опасное погружение,Pressure,2015.0,"драмы, триллеры, приключения",Великобритания,NaN,16.0,NaN,Рон Скальпелло,"Дэнни Хьюстон, Мэттью Гуд, Джо Коул, Алан МакК...",Маленькая капсула с четырьмя водолазами застря...,"выживание, подводное плавание, 2015, соединенн..."
3,656683,16228,2024.0,4,series,Содержанки,NaN,2021.0,триллеры,Россия,0.0,18.0,NaN,"Константин Богомолов, Дарья Жук, Юрий Мороз","Дарья Мороз, Софья Эрнст, Сергей Бурунов, Влад...","Тонкое исследование того, как и чем живёт стол...","Содержанки, 2021, Россия"
4,656683,6563,1568.0,5,series,Бриллианты для Джульетты,Brillianty dlya Dzhulyetty,2004.0,комедии,Россия,NaN,12.0,NaN,Валерий Чиков,"Татьяна Арнтгольц, Алиса Гребенщикова, Антон К...","Три студента — Паша, Кеша и Лева — почти не пь...",NaN
5,656683,13980,1267.0,6,film,Изгой-один: Звёздные войны. Истории.,Rogue One: A Star Wars Story,2016.0,"боевики, фантастика, приключения",США,NaN,16.0,NaN,Гарет Эвардс,"Фелисити Джонс, Диего Луна, Алан Тьюдик, Донни...",Сопротивление собирает отряд для выполнения ос...,"бунтарь, космический корабль, космическое сраж..."
6,656683,12215,1171.0,7,film,Недетское кино,Not Another Teen Movie,2001.0,"мелодрамы, комедии",США,NaN,18.0,NaN,Джоэл Галлен,"Кайлер Ли, Крис Эванс, Джейми Прессли, Эрик Кр...",Популярный в школе парень и звезда футбольной ...,"аутсайдер, мяч, поцелуй, старшая школа, школьн..."
7,656683,11505,940.0,8,series,Бездомный Бог,Noragami,2014.0,"аниме, фэнтези, приключения, комедии",Япония,NaN,12.0,NaN,"Тамура Котаро, Цуёси Хида, Сюдзи Мияхара","Хироси Камия, Маая Утида, Юки Кадзи, Аки Тоёса...",Малоизвестный Бог без собственного храма решае...,"аниме, боги, божество, духи умерших, оригиналь..."
8,656683,8491,765.0,9,film,Адский бункер,Outpost,2007.0,"боевики, ужасы, фантастика",Великобритания,NaN,18.0,NaN,Стив Баркер,"Рэй Стивенсон, Джулиан Уэдэм, Ричард Брэйк, По...",Международный отряд наемников сопровождает уче...,"бункер, нацист, восточная европа, наемник, отк..."
9,656683,3281,754.0,10,series,Вольф Мессинг: Видевший сквозь время,Messing,2009.0,"драмы, историческое",Россия,NaN,12.0,NaN,"Владимир Краснопольский, Валерий Усков","Евгений Князев, Тара Амирханова, Михаил Горево...","Мессинг родился в последний год 19 века, чтобы...",NaN


None
